In [ ]:
import pandas as pd
import numpy as np
import joblib
import os, re, argparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
STOPWORDS = set(stopwords.words('english'))
#imported required libs


In [ ]:
def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9' ]", " ", text)
    tokens = [w.lower() for w in word_tokenize(text) if w.lower() not in STOPWORDS]
    return " ".join(tokens)
#utility clean ke loiye

In [ ]:
def build_dataset(pathA, pathB):
    dfA = pd.read_csv("/Desktop/Dataset/userA_chats.csv")
    dfB = pd.read_csv("/Desktop/Dataset/userB_chats.csv")
    df = pd.concat([dfA, dfB]).sort_values(['Conversation ID', 'Timestamp']).reset_index(drop=True)
    df['Message'] = df['Message'].astype(str).apply(clean_text)
    print(f"Dataset loaded: {len(df)} messages")
    return df


In [ ]:
def build_pairs(df):
    pairs = []
    for cid, group in df.groupby('Conversation ID'):
        group = group.sort_values('Timestamp').reset_index(drop=True)
        for i in range(len(group)-1):
            cur_sender = group.loc[i, 'Sender']
            nxt_sender = group.loc[i+1, 'Sender']
            if cur_sender != nxt_sender:
                pairs.append({
                    'ConversationID': cid,
                    'question': group.loc[i, 'Message'],
                    'answer': group.loc[i+1, 'Message']
                })
    qa = pd.DataFrame(pairs)
    print(f"QA pairs extracted: {len(qa)}")
    return qa
